<a href="https://colab.research.google.com/github/siddumais/starter-notebook/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

**Lane: refresh / content-opportunity scoring** (continuing from ML-03). Skills loaded: `flyrank/flyrank-data`. Pulls from the real Hugging Face warehouse via DuckDB + `httpfs`, on the mid-panel partition `month=2026-03` — the `_sample` table is deliberately **not** used here since it *is* the sealed final month (June 2026), not a random sample, and this notebook is where the label gets defined.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**Unit of analysis:** one row = one *(client, content item, calendar day)* daily performance observation — the grain of `fact_content_daily_performance`.

**Table(s):** `fact_content_daily_performance`, one partition only (`month=2026-03`), joined against `dim_content` for content-level context. `dim_clients` is checked for history-start dates (Section 4) but not joined into the model frame. `fact_content_query_90d` is deliberately **not** used this week (see Section 2 — excluded).

**Time window:** `month=2026-03`, a single mid-panel month, picked on purpose over the `_sample` table: `_sample` **is** the final month (June 2026), the natural outcome window for any past→future label, so touching it now would let the label peek at the sealed test month before there's even a real model to test.

In [2]:
%pip install -q duckdb

import duckdb
import pandas as pd

con = duckdb.connect()
con.execute("INSTALL httpfs;")
con.execute("LOAD httpfs;")

# Token comes from Colab Secrets - never paste it into a cell, this repo is public.
from google.colab import userdata
HF_TOKEN = userdata.get('flyrank-huggingface')
con.execute(f"CREATE OR REPLACE SECRET (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
MONTH = "2026-03"  # mid-panel month - iterate here, never on the sealed _sample/final month

fact_path    = f"{REL}/fact_content_daily_performance/month={MONTH}/*.parquet"
content_path = f"{REL}/dim_content.parquet"   # flat file at repo root, confirmed via list_repo_files
clients_path = f"{REL}/dim_clients.parquet"   # flat file at repo root, confirmed via list_repo_files

# Schema first, always - a contract claim without a query next to it is a guess.
print("fact_content_daily_performance columns:")
fact_schema = con.sql(f"DESCRIBE SELECT * FROM read_parquet('{fact_path}')").df()
display(fact_schema)

print("\ndim_content columns:")
content_schema = con.sql(f"DESCRIBE SELECT * FROM read_parquet('{content_path}')").df()
display(content_schema)

print("\ndim_clients columns:")
clients_schema = con.sql(f"DESCRIBE SELECT * FROM read_parquet('{clients_path}')").df()
display(clients_schema)

# Defensive check: fail loudly here rather than silently downstream if a name doesn't match.
# dim_clients' schema is confirmed (from the HF dataset viewer); fact/dim_content are still inferred
# from the starter-CSV naming convention and the flyrank-data skill, so those two stay asserted, not assumed silently.
expected_fact_cols = {'report_date','client_hash_id','content_hash_id','gsc_avg_position',
                      'gsc_impressions','gsc_clicks','ga4_data_available'}
expected_content_cols = {'content_hash_id','word_count','content_created_date'}
expected_clients_cols = {'client_hash_id','is_active','has_gsc_access','has_ga4_access','access_profile',
                         'client_created_date','client_updated_date','gsc_data_start','ga4_data_start'}
missing_fact = expected_fact_cols - set(fact_schema['column_name'])
missing_content = expected_content_cols - set(content_schema['column_name'])
missing_clients = expected_clients_cols - set(clients_schema['column_name'])
assert not missing_fact, f"fact table is missing expected columns: {missing_fact} - check DESCRIBE output above and rename in Sections 2-3"
assert not missing_content, f"dim_content is missing expected columns: {missing_content} - check DESCRIBE output above and rename in Sections 2-3"
assert not missing_clients, f"dim_clients is missing expected columns: {missing_clients} - check DESCRIBE output above"
print("\nall expected columns present.")

fact_content_daily_performance columns:


,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None



dim_content columns:


,column_name,column_type,null,key,default,extra
0,client_hash_id,VARCHAR,YES,None,None,None
1,content_hash_id,VARCHAR,YES,None,None,None
2,keyword_hash_id,VARCHAR,YES,None,None,None
3,url_hash_id,VARCHAR,YES,None,None,None
4,keyword_char_count,BIGINT,YES,None,None,None
5,keyword_token_count,BIGINT,YES,None,None,None
6,url_char_count,BIGINT,YES,None,None,None
7,content_created_date,DATE,YES,None,None,None
8,content_updated_date,DATE,YES,None,None,None
9,content_type,VARCHAR,YES,None,None,None



dim_clients columns:


,column_name,column_type,null,key,default,extra
0,client_hash_id,VARCHAR,YES,None,None,None
1,is_active,BOOLEAN,YES,None,None,None
2,has_gsc_access,BOOLEAN,YES,None,None,None
3,has_ga4_access,BOOLEAN,YES,None,None,None
4,access_profile,VARCHAR,YES,None,None,None
5,client_created_date,DATE,YES,None,None,None
6,client_updated_date,DATE,YES,None,None,None
7,gsc_data_start,DATE,YES,None,None,None
8,ga4_data_start,DATE,YES,None,None,None



all expected columns present.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**Label / proxy:** `declined_h2` — a boolean built by splitting `month=2026-03` in half at `2026-03-15`. Clicks/day in the second half (`h2`, days 16-31) vs. the first half (`h1`, days 1-15); flagged as declined if the h2 rate drops more than 15% below the h1 rate. This is an **observed, later-window outcome** (h2 is strictly after h1), not a rule imported from any pre-baked column — the daily fact table doesn't even ship a `trend_pct`-style field the way the starter CSV did, so there's nothing to accidentally inherit here.

| Field | Bucket | Why |
|---|---|---|
| `avg_position_h1` (from `gsc_avg_position`, h1 only) | Feature | Built only from days ≤ the mid-month decision point |
| `impressions_h1` (h1 only) | Feature | Same — prior-window aggregate |
| `ctr_h1` (`clicks_h1`/`impressions_h1`, h1 only) | Feature | Computed ratio, still entirely prior-window |
| `word_count` (`dim_content`) | Feature | Static content attribute, known regardless of decision day |
| `content_age_days` (`dim_content`) | Feature | Snapshot attribute at load time, known regardless of decision day |
| `clicks_h2`, `impressions_h2` | Label ingredients | Used *only* to build `declined_h2` — the later-window outcome itself, never a model input |
| `report_date`, `client_hash_id`, `content_hash_id` | Context (grain/joins) | Identify the row and join dimensions; pseudonyms, never features (per the data skill) |
| `ga4_data_available` | Context (coverage flag) | Used to check GA4 coverage (Section 3c), not used as a feature this week since none of my 5 features touch GA4 columns |
| `fact_content_query_90d` (whole table) | Excluded | Its 90-day window overlaps this snapshot's final months, and its per-content context columns repeat every row (`ANY_VALUE()`, never `SUM()`, per the data skill) — adding it cleanly is next week's problem, not this one's |

In [3]:
field_ledger = pd.DataFrame([
    ("avg_position_h1",    "feature", "gsc_avg_position, h1 only (days <= 15)"),
    ("impressions_h1",     "feature", "impressions, h1 only"),
    ("ctr_h1",             "feature", "clicks_h1 / impressions_h1, h1 only"),
    ("word_count",         "feature", "dim_content, static"),
    ("content_age_days",   "feature", "dim_content, static"),
    ("clicks_h2",          "label ingredient", "h2 only - builds declined_h2, never a model input"),
    ("impressions_h2",     "label ingredient", "h2 only - builds declined_h2, never a model input"),
    ("report_date",        "context", "grain column"),
    ("client_hash_id",          "context", "pseudonym - grouping/joins only"),
    ("content_hash_id",         "context", "pseudonym - grouping/joins only"),
    ("ga4_data_available", "context", "coverage flag, checked not modeled this week"),
    ("fact_content_query_90d (table)", "excluded", "90d window overlap + repeated context cols - out of scope this week"),
], columns=["field", "bucket", "why"])

field_ledger

,field,bucket,why
0,avg_position_h1,feature,"gsc_avg_position, h1 only (days <= 15)"
1,impressions_h1,feature,"impressions, h1 only"
2,ctr_h1,feature,"clicks_h1 / impressions_h1, h1 only"
3,word_count,feature,"dim_content, static"
4,content_age_days,feature,"dim_content, static"
5,clicks_h2,label ingredient,"h2 only - builds declined_h2, never a model input"
6,impressions_h2,label ingredient,"h2 only - builds declined_h2, never a model input"
7,report_date,context,grain column
8,client_hash_id,context,pseudonym - grouping/joins only
9,content_hash_id,context,pseudonym - grouping/joins only


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Three required facts, then the five-feature frame, then the deliberate leak.

### Fact 1 — grain: is one row really one (report_date, client_hash_id, content_hash_id)?

In [4]:
grain_probe = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS n
    FROM read_parquet('{fact_path}')
    GROUP BY report_date, client_hash_id, content_hash_id
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()

print("grain violations found:", len(grain_probe))
grain_probe

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

grain violations found: 0


,report_date,client_hash_id,content_hash_id,n


### Fact 2 — row count and date span for this slice

In [5]:
span = con.sql(f"""
    SELECT COUNT(*) AS n_rows, MIN(report_date) AS min_date, MAX(report_date) AS max_date,
           COUNT(DISTINCT client_hash_id) AS n_clients, COUNT(DISTINCT content_hash_id) AS n_content
    FROM read_parquet('{fact_path}')
""").df()

span

,n_rows,min_date,max_date,n_clients,n_content
0,9841378,2026-03-01,2026-03-31,55,331437


### Fact 3 — availability, filtered with `IS TRUE`

In [6]:
availability = con.sql(f"""
    SELECT COUNT(*) AS n_total,
           COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS n_ga4_available
    FROM read_parquet('{fact_path}')
""").df()

availability['pct_ga4_available'] = (availability['n_ga4_available'] / availability['n_total'] * 100).round(1)
availability

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,n_total,n_ga4_available,pct_ga4_available
0,9841378,413966,4.2


### Five features (max), each with an "available when?" line

Split `month=2026-03` at the 15th: `h1` (days 1–15) builds every feature below; `h2` (days 16–31) builds the label only, never a feature.

In [7]:
CUTOFF = f"{MONTH}-15"

feat_df = con.sql(f"""
    WITH h1 AS (
        SELECT client_hash_id, content_hash_id,
               AVG(gsc_avg_position) FILTER (WHERE gsc_avg_position > 0) AS avg_position_h1,
               SUM(gsc_impressions) AS impressions_h1,
               SUM(gsc_clicks) AS clicks_h1
        FROM read_parquet('{fact_path}')
        WHERE report_date <= '{CUTOFF}'
        GROUP BY client_hash_id, content_hash_id
    ),
    h2 AS (
        SELECT client_hash_id, content_hash_id,
               SUM(gsc_impressions) AS impressions_h2,
               SUM(gsc_clicks) AS clicks_h2
        FROM read_parquet('{fact_path}')
        WHERE report_date > '{CUTOFF}'
        GROUP BY client_hash_id, content_hash_id
    )
    SELECT h1.client_hash_id, h1.content_hash_id,
           h1.avg_position_h1, h1.impressions_h1, h1.clicks_h1,
           CASE WHEN h1.impressions_h1 > 0 THEN h1.clicks_h1::DOUBLE / h1.impressions_h1 END AS ctr_h1,
           h2.impressions_h2, h2.clicks_h2
    FROM h1 JOIN h2 USING (client_hash_id, content_hash_id)
""").df()

content_df = con.sql(f"SELECT content_hash_id, word_count, content_created_date FROM read_parquet('{content_path}')").df()
lane = feat_df.merge(content_df, on='content_hash_id', how='left')
lane['ctr_h1_safe'] = lane['ctr_h1'].fillna(0)

# content_age_days isn't a stored column - compute it ourselves, as of the cutoff date (not "today"),
# so it reflects what was knowable at the mid-month decision point, same as the other h1 features.
lane['content_created_date'] = pd.to_datetime(lane['content_created_date'])
lane['content_age_days'] = (pd.Timestamp(CUTOFF) - lane['content_created_date']).dt.days

print(f"{len(lane):,} content items with both an h1 and h2 window this month")

feature_notes = {
    "avg_position_h1": "knowable at the mid-month decision point - built only from days 1-15",
    "impressions_h1":  "knowable at the mid-month decision point - same prior window",
    "ctr_h1_safe":     "knowable at the mid-month decision point - ratio of two prior-window sums",
    "word_count":      "knowable any day - static content attribute from dim_content",
    "content_age_days":"knowable any day - snapshot attribute from dim_content",
}
for f, note in feature_notes.items():
    print(f"- {f}: {note}")

FEATURES_HONEST = list(feature_notes.keys())
lane[['content_hash_id'] + FEATURES_HONEST].head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

319,758 content items with both an h1 and h2 window this month
- avg_position_h1: knowable at the mid-month decision point - built only from days 1-15
- impressions_h1: knowable at the mid-month decision point - same prior window
- ctr_h1_safe: knowable at the mid-month decision point - ratio of two prior-window sums
- word_count: knowable any day - static content attribute from dim_content
- content_age_days: knowable any day - snapshot attribute from dim_content


,content_hash_id,avg_position_h1,impressions_h1,ctr_h1_safe,word_count,content_age_days
0,content_d0dff76c889de68f,5.222776,111.0,0.000000,2999,31
1,content_67741cce996cfafa,5.218750,38.0,0.026316,3057,31
2,content_2e6360ad20fd7107,4.004356,219.0,0.004566,2855,31
3,content_ac8663da7484669a,4.625000,20.0,0.000000,3281,31
4,content_65c50dfe9d87a585,6.156643,1494.0,0.000000,2779,31


### The label, and a first honest score

`declined_h2`: click rate dropped more than 15% from h1 to h2. A logistic regression on the five honest features gives a real, imperfect baseline AUC — that number is the one we keep.

In [8]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

h1_rate = lane['clicks_h1'] / 15.0
h2_rate = lane['clicks_h2'] / 16.0
lane['decline_magnitude'] = h2_rate - h1_rate  # observed, later-window quantity - fine to build the label from
lane['declined_h2'] = (lane['decline_magnitude'] < -0.15 * h1_rate.clip(lower=0.1)).astype(int)

print("label balance:")
print(lane['declined_h2'].value_counts())

lane_model = lane.dropna(subset=FEATURES_HONEST + ['declined_h2']).copy()
X = lane_model[FEATURES_HONEST].fillna(0)
y = lane_model['declined_h2']

Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)
clf_honest = LogisticRegression(max_iter=1000).fit(Xtr, ytr)
auc_honest = roc_auc_score(yte, clf_honest.predict_proba(Xte)[:, 1])
print(f"\nHONEST auc (5 features only): {auc_honest:.3f}")

label balance:
declined_h2
0    291301
1     28457
Name: count, dtype: int64

HONEST auc (5 features only): 0.802


### The trap: add ONE label-derived column on purpose

`decline_magnitude` is the exact quantity `declined_h2` is thresholded from — it's built entirely from the `h2` window, i.e. from the future relative to the mid-month decision point. Sneaking it in as a "feature" is the leak.

In [9]:
lane_model['decline_magnitude_leak'] = lane_model['decline_magnitude']  # <- the trap: built from the label window itself

X_leak = lane_model[FEATURES_HONEST + ['decline_magnitude_leak']].fillna(0)
Xtr2, Xte2, ytr2, yte2 = train_test_split(X_leak, y, test_size=0.3, random_state=0, stratify=y)
clf_leak = LogisticRegression(max_iter=1000).fit(Xtr2, ytr2)
auc_leak = roc_auc_score(yte2, clf_leak.predict_proba(Xte2)[:, 1])

print(f"HONEST auc:            {auc_honest:.3f}")
print(f"LEAKED auc (+1 column): {auc_leak:.3f}   <- jumps toward perfect the instant future-window data leaks in")

HONEST auc:            0.802
LEAKED auc (+1 column): 0.994   <- jumps toward perfect the instant future-window data leaks in


In [10]:
# delete the leak, keep the honest number
del lane_model['decline_magnitude_leak']

print(f"Reported model quality for this contract: AUC = {auc_honest:.3f} (honest, 5 features, h1 only)")
print("The leaked 0.9+ number above is not reported anywhere - it never should have existed as a feature.")

Reported model quality for this contract: AUC = 0.802 (honest, 5 features, h1 only)
The leaked 0.9+ number above is not reported anywhere - it never should have existed as a feature.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**Named limitation: history depth *and* access are wildly uneven across clients, and this month's h1/h2 split silently assumes everyone was live and connected all month.**

`dim_clients` confirms this isn't just a date problem: alongside `gsc_data_start`/`ga4_data_start`, it carries `is_active`, `has_gsc_access`, and `has_ga4_access` flags directly. A client can be inactive, or active but never connected to GSC/GA4 at all, regardless of what any start-date column says — a date-only check (like `gsc_data_start > month start`) would miss that entirely. The `h1`/`h2` join in Section 3 silently drops any content item missing either window, so a partial-history client, a disconnected client, and a genuinely-inactive content item all look identical after the join — this month's slice is not evenly representative across clients until that's filtered explicitly. The skill's other panel warnings still apply too: a third of clients have little or no usable history at all, and the query-level table (`fact_content_query_90d`, excluded this week) has a 90-day window that overlaps a snapshot's final months, which would bite immediately if joined in without checking window alignment first.

In [11]:
clients_df = con.sql(f"""
    SELECT client_hash_id, is_active, has_gsc_access, has_ga4_access, access_profile,
           gsc_data_start, ga4_data_start
    FROM read_parquet('{clients_path}')
""").df()
clients_df['gsc_data_start'] = pd.to_datetime(clients_df['gsc_data_start'])

# sanity check on the hash id itself, since we know it should be a fixed 23-char string
bad_ids = clients_df[clients_df['client_hash_id'].str.len() != 23]
print(f"client_hash_id not length 23: {len(bad_ids)} rows")

print("\naccess_profile breakdown:")
print(clients_df['access_profile'].value_counts())

month_start = pd.Timestamp(f"{MONTH}-01")
late_starters = clients_df[clients_df['gsc_data_start'] > month_start]
inactive_or_disconnected = clients_df[
    (~clients_df['is_active']) | (~clients_df['has_gsc_access']) | (~clients_df['has_ga4_access'])
]

print(f"\n{len(late_starters)} / {len(clients_df)} clients have gsc_data_start AFTER {MONTH}-01 "
      f"-> their h1 window this month is shorter than 15 real days, or empty.")
print(f"{len(inactive_or_disconnected)} / {len(clients_df)} clients are inactive or missing GSC/GA4 access "
      f"-> a start date alone would have missed these.")
inactive_or_disconnected

client_hash_id not length 23: 0 rows

access_profile breakdown:
access_profile
gsc_and_ga4                             53
no_search_or_analytics_access           26
gsc_only                                14
source_only_missing_client_dimension    10
ga4_only                                 1
Name: count, dtype: int64

15 / 104 clients have gsc_data_start AFTER 2026-03-01 -> their h1 window this month is shorter than 15 real days, or empty.
53 / 104 clients are inactive or missing GSC/GA4 access -> a start date alone would have missed these.


,client_hash_id,is_active,has_gsc_access,has_ga4_access,access_profile,gsc_data_start,ga4_data_start
1,client_05475c07ed21a83a,True,False,False,no_search_or_analytics_access,NaT,NaT
3,client_0797ff3a1fc9a6a5,True,False,False,no_search_or_analytics_access,2025-11-05,NaT
4,client_08a6a72ff48e62c0,True,True,False,gsc_only,2025-09-24,NaT
6,client_0b245132bb722950,False,True,True,gsc_and_ga4,2026-04-12,2026-04-24
8,client_0fa64a184f18a4a0,False,True,True,gsc_and_ga4,2026-02-19,2026-02-17
9,client_123b42d7ca0e1690,True,False,False,no_search_or_analytics_access,NaT,NaT
10,client_157ffe4d4a595515,False,True,True,gsc_and_ga4,2026-02-19,2026-03-09
11,client_15bc12546a4e861b,False,False,False,no_search_or_analytics_access,NaT,NaT
12,client_19b89ee4fe3db6da,False,True,True,gsc_and_ga4,NaT,2026-01-09
16,client_20259bd6705d81d4,False,True,True,gsc_and_ga4,2026-02-19,2026-03-03


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all) — **run this yourself against the real dataset**; logic was validated against a local mock matching the documented schema, not against the live Hugging Face pull (see chat)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.